In [ ]:
import torch
import triton
import triton.language as tl
from triton.testing import do_bench

print(torch.__version__, triton.__version__, torch.cuda.get_device_name())

torch.backends.cuda.matmul.allow_tf32 = True

In [ ]:
@triton.jit
def matmul_kernel(
    a_ptr,
    b_ptr,
    c_ptr,
    M,
    N,
    K,
    stride_am,
    stride_ak,
    stride_bk,
    stride_bn,
    stride_cm,
    stride_cn,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
    num_stages: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)

    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)
    for k in tl.range(0, K, BLOCK_K, num_stages=num_stages):
        am_offsets = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
        ak_offsets = k + tl.arange(0, BLOCK_K)

        a = tl.load(
            a_ptr + am_offsets[:, None] * stride_am + ak_offsets * stride_ak,
            mask=(am_offsets < M)[:, None] & (ak_offsets < K),
            other=0.0,
        )

        bn_offsets = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
        bk_offsets = k + tl.arange(0, BLOCK_K)

        b = tl.load(
            b_ptr + bk_offsets[:, None] * stride_bk + bn_offsets * stride_bn,
            mask=(bk_offsets < K)[:, None] & (bn_offsets < N),
            other=0.0,
        )

        acc = tl.dot(a, b, acc=acc)

    cm_offsets = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    cn_offsets = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    tl.store(
        c_ptr + cm_offsets[:, None] * stride_cm + cn_offsets * stride_cn,
        acc,
        mask=(cm_offsets < M)[:, None] & (cn_offsets < N),
    )


def matmul(a, b, **kwargs):
    assert a.ndim == b.ndim == 2
    assert a.shape[1] == b.shape[0]
    assert a.device == b.device
    assert a.dtype == b.dtype

    M, K = a.shape
    _, N = b.shape

    out = torch.empty((M, N), device=a.device, dtype=a.dtype)

    BLOCK_M = kwargs.get("BLOCK_M", 128)
    BLOCK_N = kwargs.get("BLOCK_N", 128)
    BLOCK_K = kwargs.get("BLOCK_K", 32)
    num_stages = kwargs.get("num_stages", 4)

    grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))

    matmul_kernel[grid](
        a,
        b,
        out,
        M,
        N,
        K,
        a.stride(0),
        a.stride(1),
        b.stride(0),
        b.stride(1),
        out.stride(0),
        out.stride(1),
        BLOCK_M,
        BLOCK_N,
        BLOCK_K,
        num_stages,
    )

    return out

In [ ]:
a = torch.randn(8192, 128, device="cuda", dtype=torch.float16)
b = torch.randn(128, 8192, device="cuda", dtype=torch.float16)

out = matmul(a, b)
expected = a @ b

torch.testing.assert_close(out, expected)

cublas = lambda: torch.matmul(a, b)

print(f"{'block_mn':8s} {'cublas':14s} {'yours':14s}")
for size in [128, 256]:  # beyond 256 too slow
    yours = lambda: matmul(a, b, BLOCK_M=size, BLOCK_N=size)

    ms_naive = do_bench(cublas, warmup=10, rep=100)
    ms_triton = do_bench(yours, warmup=10, rep=100)
    print(f"{size:<8d} {ms_naive:<14.3f} {ms_triton:<14.3f}")


a = torch.randn(1024, 8192, device="cuda", dtype=torch.float16)
b = torch.randn(8192, 1024, device="cuda", dtype=torch.float16)

cublas = lambda: torch.matmul(a, b)


print(f"{'block_k':8s} {'naive':14s} {'yours':14s}")
for size in [16, 32, 64]:  # beyond 64 out of shared memory
    yours = lambda: matmul(a, b, BLOCK_K=size)

    ms_naive = do_bench(cublas, warmup=10, rep=100)
    ms_triton = do_bench(yours, warmup=10, rep=100)
    print(f"{size:<8d} {ms_naive:<14.3f} {ms_triton:<14.3f}")